# Week 2 Day 3 — LangGraph: Stateful, Multi-Step & Cyclical Agent Workflows

**Scenario:** Real agent workflows aren't a single loop — they branch, loop back, and need explicit control over state. Today we move to **LangGraph**, which models agents as graphs of nodes (computation steps) and edges (transitions), giving us deterministic control far beyond a plain `AgentExecutor`.

| Concept / Capability | Day 2 (`AgentExecutor`) | Day 3 (`LangGraph`) |
| :--- | :--- | :--- |
| **Control Flow** | Fixed, black-box ReAct `while` loop | Explicit DAG or cyclic graph of custom nodes |
| **State Representation** | Ephemeral list of `ToolMessage` / `ChatMessage` | Central typed schema (`TypedDict` / Pydantic) |
| **Branching & Cycles** | Prompt tricks or uncontrollable recursion | First-class conditional edges with state inspection |
| **Human-in-the-Loop** | Difficult callback hooks | Native `interrupt_before` / `interrupt_after` checkpoints |
| **Persistence & Time-Travel** | None (ephemeral) or external wrapper | Built-in checkpointers (`MemorySaver`) with snapshot replay |


In [1]:
# Setup and verify environment
%load_ext autoreload
%autoreload 2

import sys
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "workflow.py").is_file() else Path.cwd() / "week 2" / "day 3"
sys.path.insert(0, str(HERE))

from config import user_api_key, default_model, build_llm
from tools import calculator, get_weather, lookup_product_price

key = user_api_key()
if not key:
    raise RuntimeError("Missing GEMINI_API_KEY in .env file.")

print(f"Environment ready | Model: {default_model()} | Key: ...{key[-4:]}")
print("Day 2 Tools Loaded: calculator, get_weather, lookup_product_price")


Environment ready | Model: gemini-3.5-flash-lite | Key: ...p2dA
Day 2 Tools Loaded: calculator, get_weather, lookup_product_price


## Task 1: Graph Concepts & State Design

### 1.1 LangGraph Core Building Blocks Explained

1. **`StateGraph`**: The central graph compiler and runtime container. Parameterized by a state schema, it registers nodes, edges, conditional edges, entrypoints (`START`), and exit points (`END`). Calling `.compile()` turns the graph declaration into an executable `CompiledStateGraph` runnable.
2. **`Nodes`**: Python functions or Runnables that receive the current state (and optional runtime config) and return a dictionary of state updates. Nodes represent discrete phases of work (e.g. planning, tool retrieval, drafting, critique, human review, formatting).
3. **`Edges`**: Direct, deterministic links (`builder.add_edge(source, target)`) that specify unconditional transitions between nodes.
4. **`Conditional Edges`**: Dynamic routing transitions (`builder.add_conditional_edges(source, router_fn, path_map)`). The routing function inspects current state values (e.g., quality scores or retry counts) and returns a key directing execution down a specific path (including looping backwards).
5. **`Shared State Object`**: The single source of truth across the workflow. Defined as a `TypedDict` or Pydantic model, it tracks all variables, memory scratchpads, counters, and outputs. Nodes do not mutate state in-place; they return updates merged into state according to reducer semantics.

---

### 1.2 Concrete Workflow Architecture: Autonomous Market & Research Analyst

We design an **Autonomous Market & Technical Research Assistant** that:
1. Plans structured investigation steps for a business/software inquiry.
2. Retrieves factual data and performs cost calculations using Day 2 tools (`lookup_product_price`, `calculator`, `get_weather`).
3. Drafts an executive brief.
4. Critiques the draft against rigorous quality criteria. If quality is below threshold (`< 8.0`) and under `max_revisions`, it **loops back** to refine the draft.
5. Pauses at a **Human-in-the-Loop** checkpoint before publishing.
6. Publishes the verified report upon approval (or logs cancellation if rejected).

#### Graph Architecture Diagram (ASCII)

```
       [START]
          |
          v
       [ plan ] --------> Generates 3 focused sub-steps
          |
          v
      [retrieve] -------> Queries Day 2 tools (catalog, calculator, weather)
          |
          v
   +-> [generate_draft] -> Synthesizes facts & addresses previous critique
   |      |
   |      v
   |   [critique] ------> Evaluates quality score (0.0 - 10.0) & logs feedback
   |      |
   |      +---(Score < 8.0 & Revisions < Max)-------------------+
   +-------------------------------------------------------------+
          |
          +---(Score >= 8.0 or Revisions >= Max)
          |
          v
    [human_review] -----> Prepares executive review payload
          |
          v (INTERRUPT CHECKPOINT: interrupt_before=["publish"])
      [publish] --------> Validates human approval; generates signed final report
          |
          v
        [END]
```

#### Graph Architecture (Mermaid)

```mermaid
graph TD;
    __start__([Start]) --> plan[1. Plan Sub-steps];
    plan --> retrieve[2. Retrieve Tool Facts];
    retrieve --> generate_draft[3. Generate Draft];
    generate_draft --> critique[4. Critique & Score];
    critique -. "Quality < 8.0 & count < max (Loop Back)" .-> generate_draft;
    critique -. "Quality >= 8.0 or count >= max" .-> human_review[5. Human Review];
    human_review -->|Pause / Checkpoint| publish[6. Publish Executive Report];
    publish --> __end__([End]);
```


In [2]:
from typing import List, TypedDict

class ResearchWorkflowState(TypedDict, total=False):
    """Shared state schema for the research and publication workflow."""
    query: str                       # User prompt / research inquiry
    plan: List[str]                  # Planned investigation steps
    research_notes: List[str]        # Factual notes retrieved from tools
    draft: str                       # Draft report content
    critique: str                    # Feedback from critique evaluation
    quality_score: float             # Evaluated quality score (0.0 to 10.0)
    revision_count: int              # Number of critique/revision cycles completed
    max_revisions: int               # Upper guardrail to prevent infinite loops
    revision_logs: List[str]         # Historical log of scores and feedback
    human_approval_status: str       # 'pending', 'approved', 'rejected'
    human_feedback: str              # Reviewer comments or instructions
    final_report: str                # Formatted, approved deliverable

print("ResearchWorkflowState defined with fields:")
for field, ftype in ResearchWorkflowState.__annotations__.items():
    print(f" - {field:22}: {ftype}")


ResearchWorkflowState defined with fields:
 - query                 : <class 'str'>
 - plan                  : typing.List[str]
 - research_notes        : typing.List[str]
 - draft                 : <class 'str'>
 - critique              : <class 'str'>
 - quality_score         : <class 'float'>
 - revision_count        : <class 'int'>
 - max_revisions         : <class 'int'>
 - revision_logs         : typing.List[str]
 - human_approval_status : <class 'str'>
 - human_feedback        : <class 'str'>
 - final_report          : <class 'str'>


## Task 2: Build a Linear Graph

We construct a 4-node linear graph:
`START` → `plan` → `retrieve` → `generate_draft` → `publish` → `END`

- **`plan`**: Breaks the inquiry into 3 actionable steps using Gemini.
- **`retrieve`**: Uses Day 2 tools (`lookup_product_price`, `calculator`, `get_weather`) to gather verified data points into `research_notes`.
- **`generate_draft`**: Synthesizes the facts and plan into an executive brief.
- **`publish`**: Formats the final deliverable.

We compile the graph and execute it with `stream(..., stream_mode="updates")`, printing state updates after each step to verify state evolution.


In [3]:
from workflow import build_linear_graph

linear_graph = build_linear_graph()
print("Linear Graph compiled successfully.")

sample_query = "Compare Starter vs Pro plans for a team of 10 users with annual budget calculations."
print(f"\nExecuting linear graph on query: '{sample_query}'\n" + "="*70)

initial_state = {
    "query": sample_query,
    "human_approval_status": "approved"
}

# Stream and print state updates after each node execution
for update in linear_graph.stream(initial_state, stream_mode="updates"):
    for node_name, diff in update.items():
        print(f"\n[NODE COMPLETED]: {node_name.upper()}")
        for k, v in diff.items():
            if isinstance(v, list):
                print(f"  + {k} ({len(v)} items):")
                for item in v[:3]:
                    print(f"     * {item}")
            elif isinstance(v, str) and len(v) > 120:
                print(f"  + {k}: {v[:110]}... [truncated {len(v)} chars]")
            else:
                print(f"  + {k}: {v}")


Linear Graph compiled successfully.

Executing linear graph on query: 'Compare Starter vs Pro plans for a team of 10 users with annual budget calculations.'


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



[NODE COMPLETED]: PLAN
  + plan (3 items):
     * Gather Starter and Pro pricing for 10 users with annual billing discounts
     * Calculate total annual cost differences including potential add-ons
     * Assess feature limitations of Starter versus Pro for a 10-person team
  + revision_count: 0
  + max_revisions: 2
  + revision_logs (0 items):
  + human_approval_status: pending

[NODE COMPLETED]: RETRIEVE
  + research_notes (3 items):
     * [Catalog Lookup] Starter: $19/month (basic) — Single user, 5 GB storage, email support.
     * [Catalog Lookup] Pro: $49/month (standard) — Up to 10 users, 100 GB storage, priority support.
     * [Calculator Calculation] Pro plan annual cost: $49/mo * 12 = $588/year



[NODE COMPLETED]: GENERATE_DRAFT
  + draft: ### Executive Summary
This executive brief compares the Starter and Pro plans for a 10-person team, incorporat... [truncated 1961 chars]

[NODE COMPLETED]: PUBLISH
  + final_report: # PUBLICATION ABORTED

**Status:** REJECTED / HALTED BY REVIEWER
**Reviewer Feedback:** Rejected by human oper... [truncated 2110 chars]


## Task 3: Add Conditional Edges & Cycles (Self-Correction Loop)

### 3.1 Self-Correction Architecture

We introduce a **`critique`** node following `generate_draft`. The critique node evaluates the draft on quality, factual depth, and calculation precision:
- If `quality_score < 8.0` and `revision_count < max_revisions`: The conditional edge routes **back to `generate_draft`**, feeding the critique feedback into the prompt for targeted revision.
- If `quality_score >= 8.0` or `revision_count >= max_revisions`: Execution routes forward to `publish`.
- A **`max_revisions`** counter (set to 2) acts as a strict guardrail against infinite loops.
- An audit list **`revision_logs`** records each pass, score, and decision.

---

### 3.2 Why Loop-Back is Hard in AgentExecutor but Natural in LangGraph

> **Explanation:**
> In **`AgentExecutor`**, the control loop is an opaque, hardcoded ReAct cycle (`Thought -> Action -> Observation -> Action`). There is no first-class representation of discrete operational phases (such as Planning vs. Drafting vs. Critique), no mechanism to branch on custom quality metrics, and no ability to roll back the model's scratchpad to re-invoke an upstream reasoning step. Attempting self-correction in `AgentExecutor` requires fragile prompt tricks that force the LLM to remember to critique itself in a single conversational thread.
> 
> In **`LangGraph`**, cyclical execution is a native graph topology. Nodes are standard Python functions, and transitions are explicit directed edges. A conditional edge can inspect any field in the shared `State` object (e.g., `state["quality_score"] < 8.0`) and route back to any upstream node (`generate_draft`), passing accumulated critique and revision counts deterministically without modifying the underlying framework.


In [4]:
from workflow import build_cyclical_graph

cyclical_graph = build_cyclical_graph()
print("Cyclical Graph compiled successfully.")

query_cyclical = "Evaluate Slack Pro vs Notion Plus pricing for a growing team of 20 members."
print(f"\nExecuting cyclical self-correction graph on query: '{query_cyclical}'\n" + "="*70)

input_state = {
    "query": query_cyclical,
    "max_revisions": 2,
    "human_approval_status": "approved"
}

step_counter = 1
for update in cyclical_graph.stream(input_state, stream_mode="updates"):
    for node_name, diff in update.items():
        print(f"\nStep {step_counter}: -> Node [{node_name.upper()}]")
        step_counter += 1
        if "quality_score" in diff:
            print(f"   Quality Score: {diff['quality_score']}/10.0 | Revision Count: {diff['revision_count']}")
            print(f"   Critique: {diff['critique'][:120]}...")
        if "revision_logs" in diff and diff["revision_logs"]:
            print(f"   Audit Log: {diff['revision_logs'][-1]}")
        if "draft" in diff:
            print(f"   Draft Length: {len(diff['draft'])} characters")


Cyclical Graph compiled successfully.

Executing cyclical self-correction graph on query: 'Evaluate Slack Pro vs Notion Plus pricing for a growing team of 20 members.'



Step 1: -> Node [PLAN]

Step 2: -> Node [RETRIEVE]



Step 3: -> Node [GENERATE_DRAFT]
   Draft Length: 2107 characters

Step 4: -> Node [CRITIQUE]
   Quality Score: 6.5/10.0 | Revision Count: 1
   Critique: Critique Pass 1: The draft has good structure, but lacks a side-by-side cost breakdown, explicit annual savings analysis...
   Audit Log: Pass 1: Score 6.5/10.0 -> Below 8.0 threshold. Looping back to generate_draft.



Step 5: -> Node [GENERATE_DRAFT]
   Draft Length: 2831 characters



Step 6: -> Node [CRITIQUE]
   Quality Score: 9.5/10.0 | Revision Count: 2
   Critique: This revised research draft features exceptionally clear pricing breakdowns, accurate financial calculations, and a well...
   Audit Log: Pass 2: Score 9.5/10.0 -> Quality threshold met (>= 8.0). Proceeding to review.

Step 7: -> Node [PUBLISH]


## Task 4: Human-in-the-Loop & Interrupts

### 4.1 Checkpoint Interrupts
Autonomous agents operating in production must be gated before executing **irreversible, financial, legal, or high-blast-radius actions** (e.g., sending client communications, purchasing licenses, deploying code).

LangGraph provides native human-in-the-loop support via **checkpointers** and **`interrupt_before` / `interrupt_after`**:
1. When the graph reaches a node specified in `interrupt_before=["publish"]`, execution safely pauses and commits the full state snapshot to the checkpointer (`MemorySaver`).
2. A human operator inspects `graph.get_state(config)`.
3. The human provides approval or rejection via `graph.update_state(...)`.
4. The graph is resumed seamlessly using `graph.invoke(None, config=config)`.

---

### 4.2 Product Discussion: When to Require HITL vs. Full Autonomy

| Dimension | Human-in-the-Loop Required | Full Autonomy Acceptable |
| :--- | :--- | :--- |
| **Action Reversibility** | Irreversible / destructive writes: email dispatch, DB deletion, financial transactions | Idempotent / read-only operations: data retrieval, web search, internal drafting |
| **Financial & Legal Stakes** | High: signing contracts, approving credit limits, spending budget | Low: calculating currency conversions, generating internal summaries |
| **Model Confidence** | Borderline quality scores (`< 8.0`), out-of-distribution prompts | High quality scores (`>= 9.0`), routine repetitive classification |
| **Regulatory & Compliance** | HIPAA, GDPR, customer-facing advice with liability | Internal sandboxed pipelines, developer tooling with code review |


In [5]:
from langgraph.checkpoint.memory import MemorySaver
from workflow import build_full_workflow_graph

# Initialize persistent in-memory checkpointer
memory = MemorySaver()
full_graph = build_full_workflow_graph(checkpointer=memory, interrupt_before=["publish"])

# -------------------------------------------------------------------------
# SCENARIO A: Human Approval Workflow
# -------------------------------------------------------------------------
print("="*70)
print("SCENARIO A: Human Approval Demonstration")
print("="*70)

thread_a = {"configurable": {"thread_id": "thread-approval-demo"}}

# 1. Run until interrupt
print("1. Launching graph execution until interrupt checkpoint...")
full_graph.invoke({
    "query": "Procurement analysis for GitHub Copilot Business (15 seats) and Slack Pro.",
}, config=thread_a)

# Inspect paused state
state_a = full_graph.get_state(thread_a)
print(f"\nGraph paused! Next pending node: {state_a.next}")
print(f"Current Draft chars: {len(state_a.values['draft'])}")
print(f"Evaluated Quality Score: {state_a.values['quality_score']}/10.0")
print(f"Completed revisions: {state_a.values['revision_count']}")

# 2. Simulate human operator approval
print("\n2. Human Operator reviews draft and grants APPROVAL...")
full_graph.update_state(thread_a, {
    "human_approval_status": "approved",
    "human_feedback": "Approved by VP of Engineering: budget verified for Q3."
})

# 3. Resume graph
print("3. Resuming graph execution with None input...")
res_a = full_graph.invoke(None, config=thread_a)
print("\nFinal Published Report Excerpt:\n" + "-"*50)
print(res_a["final_report"][:450] + "\n...")


# -------------------------------------------------------------------------
# SCENARIO B: Human Rejection Workflow
# -------------------------------------------------------------------------
print("\n" + "="*70)
print("SCENARIO B: Human Rejection Demonstration")
print("="*70)

thread_b = {"configurable": {"thread_id": "thread-rejection-demo"}}

# 1. Run until interrupt
print("1. Launching graph execution for second thread...")
full_graph.invoke({
    "query": "Enterprise tier licensing quote for immediate purchase.",
}, config=thread_b)

# 2. Simulate human operator rejection
print("\n2. Human Operator reviews draft and REJECTS publication...")
full_graph.update_state(thread_b, {
    "human_approval_status": "rejected",
    "human_feedback": "Rejected: Quote exceeds procurement spending threshold. Request vendor discount."
})

# 3. Resume graph
print("3. Resuming graph execution with rejection...")
res_b = full_graph.invoke(None, config=thread_b)
print("\nFinal Output on Rejection:\n" + "-"*50)
print(res_b["final_report"][:350] + "\n...")


SCENARIO A: Human Approval Demonstration
1. Launching graph execution until interrupt checkpoint...



Graph paused! Next pending node: ('publish',)
Current Draft chars: 3402
Evaluated Quality Score: 9.5/10.0
Completed revisions: 2

2. Human Operator reviews draft and grants APPROVAL...
3. Resuming graph execution with None input...

Final Published Report Excerpt:
--------------------------------------------------
# EXECUTIVE REPORT (PUBLISHED)

**Status:** APPROVED FOR RELEASE
**Quality Score:** 9.5/10.0 (after 2 revision cycles)
**Human Reviewer Feedback:** Approved by VP of Engineering: budget verified for Q3.

## Audit Trail
  - Pass 1: Score 6.5/10.0 -> Below 8.0 threshold. Looping back to generate_draft.
  - Pass 2: Score 9.5/10.0 -> Quality threshold met (>= 8.0). Proceeding to review.

## Final Content

# Executive Brief: Procurement Analysis for G
...

SCENARIO B: Human Rejection Demonstration
1. Launching graph execution for second thread...



2. Human Operator reviews draft and REJECTS publication...
3. Resuming graph execution with rejection...

Final Output on Rejection:
--------------------------------------------------
# PUBLICATION ABORTED

**Status:** REJECTED / HALTED BY REVIEWER
**Reviewer Feedback:** Rejected: Quote exceeds procurement spending threshold. Request vendor discount.
**Draft Retained for Review:**

**EXECUTIVE BRIEF: Enterprise Tier Licensing Quote**

### 1. Executive Summary
This brief provides an immediate purchase quote and strategic financia
...


## Task 5: Persistence & Debugging

### 5.1 Durable Checkpointing & State History
Because LangGraph commits state to a checkpointer (`MemorySaver`, SQLite, or PostgreSQL) at every node boundary, we can:
1. **Resume paused sessions across runs**: Thread configurations (`thread_id`) uniquely isolate sessions, allowing paused conversations to resume seamlessly at any point.
2. **Inspect complete execution history**: `graph.get_state_history(config)` exposes full chronological snapshots of state at each step.
3. **Time-Travel & Branch**: Fork execution from an earlier checkpoint to debug or explore alternative scenarios without re-running the entire workflow.

---

### 5.2 LangChain AgentExecutor vs. LangGraph: Decision Guide

| Evaluation Axis | LangChain `AgentExecutor` | `LangGraph` |
| :--- | :--- | :--- |
| **Architecture** | Single-loop agent runner | State machine / directed graph |
| **Control Flow** | Rigid, implicit while-loop | Explicit nodes and user-defined edges |
| **State Handling** | Untyped message list | Strongly-typed schema (`TypedDict`/Pydantic) |
| **Cycles & Self-Correction** | Cannot easily loop back to specific steps | First-class cyclic edges with counter limits |
| **Human-in-the-Loop** | Awkward callback interception | Native checkpointer interrupts (`interrupt_before/after`) |
| **Persistence** | External session memory wrappers | First-class Checkpointers (`MemorySaver`, Sqlite, Postgres) per thread |
| **Time-Travel Debug** | Not supported; cannot rewind | State snapshots; fork execution from any past checkpoint |
| **When to Use** | Quick prototypes, simple single-prompt Q&A | Complex multi-step production pipelines, cyclic agents, HITL workflows |


In [6]:
# Demonstrate Persistence Across Sessions, State History, and Time-Travel Replay

# -------------------------------------------------------------------------
# Part A: Multi-Session Persistence and Resuming Paused Conversation
# -------------------------------------------------------------------------
print("="*70)
print("PART A: Multi-Session Persistence & Resuming Paused Thread")
print("="*70)

thread_c = {"configurable": {"thread_id": "thread-persistence-session"}}

# 1. Run session until paused at interrupt checkpoint
print("1. Running session until paused at interrupt checkpoint...")
full_graph.invoke({
    "query": "Evaluate Starter vs Pro pricing and features for an engineering team.",
}, config=thread_c)

# 2. Inspect thread persistence across independent invocations
paused_session = full_graph.get_state(thread_c)
print(f"Session '{thread_c['configurable']['thread_id']}' state persisted:")
print(f" - Next pending node to execute: {paused_session.next}")
print(f" - State variables stored: {len(paused_session.values.keys())} keys")
print(f" - Draft preview: {paused_session.values.get('draft', '')[:100]}...\n")

# 3. Resume paused session to final publish
print("2. Resuming the paused session to completion with approval...")
full_graph.update_state(thread_c, {
    "human_approval_status": "approved",
    "human_feedback": "Approved by Product Owner for Q3 rollout."
})
resumed_session = full_graph.invoke(None, config=thread_c)
print("Resumed session completed successfully!")
print("Final Report Status Excerpt:", resumed_session.get("final_report", "")[:160].replace('\n', ' '))


# -------------------------------------------------------------------------
# Part B: State History Inspection and Time-Travel Forking
# -------------------------------------------------------------------------
print("\n" + "="*70)
print("PART B: State History Inspection & Time-Travel Debugging")
print("="*70)

thread_d = {"configurable": {"thread_id": "thread-timetravel-session"}}

# Run a run for time-travel inspection
full_graph.invoke({
    "query": "Analyze Pro plan features.",
}, config=thread_d)

# 1. Retrieve full state history timeline
history = list(full_graph.get_state_history(thread_d))
print(f"Total historical checkpoints saved for thread: {len(history)}")
print("\nHistorical Checkpoint Timeline (most recent to oldest):")
for i, snap in enumerate(history):
    cid = snap.config["configurable"]["checkpoint_id"]
    next_node = snap.next or "END"
    step_keys = list(snap.values.keys())
    print(f"  [{i}] Checkpoint {cid[:12]}... | Next: {str(next_node):18} | Keys in state: {len(step_keys)}")

# 2. Time-Travel Forking: Find checkpoint right before critique pass 1
checkpoint_before_critique = None
for snap in history:
    if snap.next == ("critique",):
        checkpoint_before_critique = snap
        break

if checkpoint_before_critique:
    target_config = checkpoint_before_critique.config
    print(f"\nFound historical checkpoint right before 'critique': {target_config['configurable']['checkpoint_id']}")
    
    # Fork state: inject updated draft at this exact point in time
    print("Time-Traveling: Editing draft state at historical checkpoint...")
    updated_cfg = full_graph.update_state(target_config, {
        "draft": "[TIME-TRAVEL FORKED DRAFT]: Pro plan ($49/mo) includes priority support and 100 GB storage for up to 10 users."
    })
    
    # Inspect forked state at the new checkpoint
    forked_state = full_graph.get_state(updated_cfg)
    print("Forked checkpoint ID:", updated_cfg["configurable"]["checkpoint_id"][:12])
    print("Forked state draft excerpt:", forked_state.values.get("draft", "")[:120])


PART A: Multi-Session Persistence & Resuming Paused Thread
1. Running session until paused at interrupt checkpoint...


Session 'thread-persistence-session' state persisted:
 - Next pending node to execute: ('publish',)
 - State variables stored: 11 keys
 - Draft preview: **Executive Brief: Starter vs. Pro Tier Evaluation for Engineering Teams**

---

### 1. Executive Su...

2. Resuming the paused session to completion with approval...
Resumed session completed successfully!
Final Report Status Excerpt: # EXECUTIVE REPORT (PUBLISHED)  **Status:** APPROVED FOR RELEASE **Quality Score:** 9.5/10.0 (after 2 revision cycles) **Human Reviewer Feedback:** Approved by 

PART B: State History Inspection & Time-Travel Debugging


Total historical checkpoints saved for thread: 9

Historical Checkpoint Timeline (most recent to oldest):
  [0] Checkpoint 1f1ac418-6bf... | Next: ('publish',)       | Keys in state: 11
  [1] Checkpoint 1f1ac418-6bf... | Next: ('human_review',)  | Keys in state: 10
  [2] Checkpoint 1f1ac418-559... | Next: ('critique',)      | Keys in state: 10
  [3] Checkpoint 1f1ac418-240... | Next: ('generate_draft',) | Keys in state: 10
  [4] Checkpoint 1f1ac418-238... | Next: ('critique',)      | Keys in state: 8
  [5] Checkpoint 1f1ac417-e34... | Next: ('generate_draft',) | Keys in state: 7
  [6] Checkpoint 1f1ac417-e34... | Next: ('retrieve',)      | Keys in state: 6
  [7] Checkpoint 1f1ac417-c78... | Next: ('plan',)          | Keys in state: 1
  [8] Checkpoint 1f1ac417-c78... | Next: ('__start__',)     | Keys in state: 0

Found historical checkpoint right before 'critique': 1f1ac418-559e-6dd4-8005-31af7583151f
Time-Traveling: Editing draft state at historical checkpoint...
Forked checkpoint ID: 

## Summary & Deliverables Completed

1. **Task 1: Graph Concepts & State Design**
   - Core primitives explained: `StateGraph`, `Nodes`, `Edges`, `Conditional Edges`, and `Shared State`.
   - Typed schema `ResearchWorkflowState` designed with 12 fields tracking plans, notes, drafts, critiques, and audit logs.
   - ASCII and Mermaid diagrams constructed.
2. **Task 2: Linear Graph**
   - 4-node pipeline implemented integrating Day 2 tools (`calculator`, `lookup_product_price`, `get_weather`).
   - Streamed execution with step-by-step state logging.
3. **Task 3: Conditional Edges & Cycles**
   - `critique` node with self-correction loop back to `generate_draft`.
   - `max_revisions` counter and `revision_logs` safeguarding against infinite cycles.
   - 2-3 sentence analysis of why loop-backs are natural in LangGraph vs rigid `AgentExecutor`.
4. **Task 4: Human-in-the-Loop & Interrupts**
   - Implemented `interrupt_before=["publish"]` with `MemorySaver`.
   - Tested both simulated approval and rejection execution paths.
   - Analyzed criteria for HITL vs full autonomy in enterprise products.
5. **Task 5: Persistence & Debugging**
   - Configured `MemorySaver` checkpointer with `thread_id` session isolation.
   - Inspected state history timeline and demonstrated time-travel checkpoint forking.
   - Complete decision table comparing `AgentExecutor` vs `LangGraph`.
